In [2]:
!pip install transformers torch scikit-learn pandas numpy

In [45]:
!pip uninstall transformers accelerate tokenizers -y
!pip cache purge


Found existing installation: transformers 4.46.2
Uninstalling transformers-4.46.2:
  Successfully uninstalled transformers-4.46.2
Found existing installation: accelerate 1.0.1
Uninstalling accelerate-1.0.1:
  Successfully uninstalled accelerate-1.0.1
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3
Files removed: 0


In [46]:
import shutil, os, sys

# Remove any old transformers folders
paths = [p for p in sys.path if 'site-packages' in p]
for p in paths:
    tf_path = os.path.join(p, "transformers")
    if os.path.exists(tf_path):
        print("Deleting:", tf_path)
        shutil.rmtree(tf_path, ignore_errors=True)

# Also clear pycache folders
for p in paths:
    for root, dirs, files in os.walk(p):
        for d in dirs:
            if "__pycache__" in d:
                shutil.rmtree(os.path.join(root, d), ignore_errors=True)


In [47]:
!pip install --no-cache-dir transformers==4.46.2 accelerate==1.0.1 torch torchvision torchaudio scikit-learn


     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.1 kB ? eta -:--:--
     ------------------ -------------------- 20.5/44.1 kB 73.1 kB/s eta 0:00:01
     ------------------ -------------------- 20.5/44.1 kB 73.1 kB/s eta 0:00:01
     --------------------------- ----------- 30.7/44.1 kB 81.9 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/44.1 kB 103.4 kB/s eta 0:00:01
     -------------------------------------- 44.1/44.1 kB 103.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --------------------

In [48]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, EarlyStoppingCallback, TrainingArguments
import torch
from torch.utils.data import Dataset
from sklearn.metrics import classification_report

ImportError: cannot import name 'is_quanto_available' from 'transformers.utils' (C:\Users\JAYASHREE\anaconda3\Lib\site-packages\transformers\utils\__init__.py)

In [ ]:
import transformers, os
print("Transformers version:", transformers.__version__)
print("Loaded from:", os.path.dirname(transformers.__file__))


In [ ]:
data = pd.read_csv("sentence_entity_sentiment.csv")

In [ ]:
#removing null values
data = data[data['sentiment']!="none"] 
# in this dataset we get null values when sentiment is none so removing that
print(data.head(10))

In [ ]:
# Y- output
y = data['sentiment']
print("Output: \n",y[:10])

In [ ]:
# label encoding 
Sentiment_mappings = {
    "Positive" : 2,
    "Neutral" : 1,
    "Negative": 0, 
    "Mixed": 1
}

In [30]:
y_mapped = pd.DataFrame(y.map(Sentiment_mappings).values, columns=["Sentiment"])

In [31]:
print(y_mapped)

     Sentiment
0            1
1            0
2            1
3            2
4            1
..         ...
137          1
138          1
139          2
140          1
141          0

[142 rows x 1 columns]


In [32]:
# Load FinBERT and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
sentimentModel = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 3)

ImportError: cannot import name 'is_quanto_available' from 'transformers.utils' (C:\Users\JAYASHREE\anaconda3\Lib\site-packages\transformers\utils\__init__.py)

In [33]:
encodings = tokenizer(
    data["sentence"].tolist(),
    data["entity"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

In [34]:
encodings["input_ids"].shape  # should be (num_samples, 128)

torch.Size([142, 37])

In [35]:
train_texts, temp_texts, train_entities, temp_entities, y_train, y_temp = train_test_split(data["sentence"].tolist(),
                                                                                                     data["entity"].tolist(), 
                                                                                                     y_mapped, 
                                                                                                     test_size=0.3,
                                                                                                     random_state = 42,
                                                                                                     stratify=y_mapped
                                                                                                    )

In [36]:
val_texts, test_texts, val_entities, test_entities, y_val, y_test = train_test_split(temp_texts,
                                                                                               temp_entities,
                                                                                               y_temp,
                                                                                               test_size = 0.5,
                                                                                               random_state = 42,
                                                                                               stratify=y_temp
                                                                                              )

In [37]:
train_encodings = tokenizer(train_texts, train_entities, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, val_entities, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, test_entities, truncation=True, padding=True, max_length=128)

In [38]:
train_labels = torch.tensor(y_train.values)
val_labels = torch.tensor(y_val.values)
test_labels = torch.tensor(y_test.values)


NameError: name 'torch' is not defined

In [ ]:
class FinDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FinDataset(train_encodings, y_train.values)
val_dataset = FinDataset(val_encodings, y_val.values)
test_dataset = FinDataset(test_encodings, y_test.values)
#now randomly we are splitting if accuracy is less use index splitting so that all entities belonging to a sentence fall in one category.

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1) # for confidence scores remove argmax
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')  # weighted handles class imbalance
    return {"accuracy": acc, "f1": f1}


In [ ]:

training_args = TrainingArguments(
    output_dir="./finbert_results",
    eval_strategy="steps",  
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
)


In [ ]:

trainer = Trainer(
    model=sentimentModel,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop if no improvement in 3 evals
)


In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(test_dataset)

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)
print(classification_report(y_test, pred_labels, target_names=["Negative", "Neutral", "Positive"]))

In [ ]:
trainer.save_model("./finbert-entity-sentiment")
tokenizer.save_pretrained("./finbert-entity-sentiment")